# HW2 - ATP  網球記錄分析

### 資料來源說明與欄位定義 (Data Dictionary & Citation)

### 1. 資料引用
分析採用 Jeff Sackmann 開源 ATP 網球資料集：
原始資料庫：Jeff Sackmann / Tennis ATP Data (GitHub: `https://github.com/JeffSackmann/tennis_atp`)
本作業使用的固定 repository：`https://github.com/Haz22537/JEFFSM_tennis_atp`

### 2. 關鍵欄位定義與命名對照
* **欄位命名備註**：Jeff Sackmann 資料庫之 README 記載之欄位名稱（如 `first_name`、`birth_date`、`country_code`）與實際 CSV 檔案存在微幅差異。分析均以實際 CSV 檔案中之標準欄位名稱（如 `name_first`、`dob`、`ioc`）為準進行資料處理與統計。
* **`tourney_level` (比賽層級)**:
  - `G`: Grand Slam (大滿貫賽事)
  - `M`: Masters 1000 (ATP 1000 大師賽)
  - `A`: Other Tour-Level (ATP 500 / 250 常規巡迴賽)
  - `F`: Tour Finals (ATP 年終總決賽 / NextGen Finals)
  - `D`: Davis Cup (戴維斯盃國際團體賽)
  - `O`: Olympics (奧林匹克運動會網球賽事)
* **`dob` (出生日期)**: 格式為 8 碼整數 `YYYYMMDD`。本分析之「不完整」定義涵蓋缺失空值 (`NaN`)、非 8 碼之部分數值以及無法轉為合法月日之無效日期（共計 18,504 筆）。
* **`winner_ht` / `loser_ht` (身高)**: 單位為公分 (cm)。統計分析中已剔除任一方為缺失值 (`NaN`) 之 148 場賽事，有效比較場次共 11,564 場。

## Q1:讀取 atp_players.csv

- 裡面有哪些欄位？
- 總共有多少球員在列表中？
- 全部球員中，有多少 dob 生日資料不完整?
- 依照 ioc 國籍來分類，有最多球員的國家有哪些?
- TPE  和 TWN 各有多少球員？
- TPE 的球員中，有包含哪些姓氏？

In [1]:
import os
import pandas as pd

# 自動判斷資料路徑
DATA_DIR = "JEFFSM_tennis_atp" if os.path.exists("JEFFSM_tennis_atp/atp_players.csv") else "."
players_path = os.path.join(DATA_DIR, "atp_players.csv")

# 加上 low_memory=False 消除 DtypeWarning
players = pd.read_csv(players_path, low_memory=False)

# 1. 欄位有哪些？
print(f"1. 欄位名稱 (共 {len(players.columns)} 個):")
print(list(players.columns))
print()

# 2. 總共有多少球員？
print(f"2. 總球員數: {len(players):,} 位\n")

# 3. 全部球員中，有多少 dob 生日資料不完整？
# (1) 去除浮點數讀取產生的小數點 .0 並轉為乾淨字串
dob_clean_str = players["dob"].astype("string").str.replace(r"\.0$", "", regex=True).str.strip()

# (2) 嚴格匹配 8 碼純數字 (YYYYMMDD)
valid_8digit_format = dob_clean_str.str.fullmatch(r"\d{8}", na=False)

# (3) 轉為真實日期物件，無法轉成合法日期的會被強制變為 NaT
valid_dates = pd.to_datetime(dob_clean_str.where(valid_8digit_format), format="%Y%m%d", errors="coerce")

# (4) 統計不完整數量（包含原本為空值、不足/超過8碼、以及無效日期）
incomplete_dob_count = valid_dates.isna().sum()
nan_only_count = players["dob"].isna().sum()

print(f"3. dob 生日資料不完整的球員數: {incomplete_dob_count:,} 位")
print(f"   (備註：其中完全缺失空值 NaN 為 {nan_only_count:,} 位，格式不全或無效日期為 {incomplete_dob_count - nan_only_count:,} 位)\n")

# 4. 依照 ioc 分類，球員最多的國家 (Top 10)
print("4. 球員最多的前 10 個國家 (IOC):")
print(players["ioc"].value_counts().head(10))
print()

# 5. TPE 和 TWN 各有多少球員？
tpe_count = (players["ioc"] == "TPE").sum()
twn_count = (players["ioc"] == "TWN").sum()
print(f"5. 國籍數量 -> TPE: {tpe_count} 位 | TWN: {twn_count} 位\n")

# 6. TPE 球員中包含哪些姓氏？
tpe_players = players[players["ioc"] == "TPE"]
tpe_surnames = sorted(tpe_players["name_last"].dropna().unique().tolist())
print(f"6. TPE 球員包含的姓氏 (共 {len(tpe_surnames)} 個):")
print(tpe_surnames)

1. 欄位名稱 (共 8 個):
['player_id', 'name_first', 'name_last', 'hand', 'dob', 'ioc', 'height', 'wikidata_id']

2. 總球員數: 65,989 位

3. dob 生日資料不完整的球員數: 18,504 位
   (備註：其中完全缺失空值 NaN 為 18,402 位，格式不全或無效日期為 102 位)

4. 球員最多的前 10 個國家 (IOC):
ioc
USA    13228
AUS     3285
GBR     3281
ESP     3081
GER     2742
ITA     2726
FRA     2648
BRA     2104
ARG     1760
MEX     1342
Name: count, dtype: int64

5. 國籍數量 -> TPE: 201 位 | TWN: 1 位

6. TPE 球員包含的姓氏 (共 64 個):
['Ballout', 'Chan', 'Chang', 'Chen', 'Cheng', 'Chiang', 'Chien', 'Chiu', 'Chu', 'Chuang', 'Chung', 'Fu', 'Han', 'He', 'Ho', 'Hou', 'Hsieh', 'Hsu', 'Hsueh', 'Hu', 'Huang', 'Hung', 'Jao', 'Jung', 'Kao', 'Ko', 'Kuo', 'Lai', 'Lee', 'Li', 'Liang', 'Liao', 'Lien', 'Lin', 'Liu', 'Lo', 'Lu', 'Mao', 'Meng', 'Menn', 'Ou', 'Pai', 'Pan', 'Peng', 'Shen', 'Shie', 'Shieh', 'Shih', 'Su', 'Sung', 'Tan', 'Tang', 'Tsai', 'Tseng', 'Wang', 'Wei', 'Wen', 'Wong', 'Wu', 'Yang', 'Yeh', 'Yi', 'Yin', 'Yu']


## Q2:讀取 atp_matches*202[1234].csv” 

- TPE 球員贏了多少場？輸了多少場？
- 有多少種比賽層級？
- 有多少場的 winner 身高比 loser 高/相等/低？
    - 依照不同的 tourney_level 來統計身高和勝負間的關係

In [2]:
import os
import glob
import pandas as pd

# 1. 自動判斷資料目錄並讀取 2021 至 2024 年賽事 CSV
DATA_DIR = "JEFFSM_tennis_atp" if os.path.exists("JEFFSM_tennis_atp/atp_players.csv") else "."
match_files = sorted(glob.glob(os.path.join(DATA_DIR, "atp_matches_202[1-4].csv")))
print(f"讀取的比賽檔案: {[os.path.basename(f) for f in match_files]}")

matches = pd.concat([pd.read_csv(f, low_memory=False) for f in match_files], ignore_index=True)
print(f"總比賽場次: {len(matches):,} 場\n")

# ==============================================================================
# 2. TPE 球員贏了多少場？輸了多少場？
# ==============================================================================
tpe_wins = (matches["winner_ioc"] == "TPE").sum()
tpe_losses = (matches["loser_ioc"] == "TPE").sum()

print("=" * 60)
print(f"TPE 球員勝負統計:")
print(f"- 勝場: {tpe_wins} 場")
print(f"- 敗場: {tpe_losses} 場")
print("=" * 60)

# ==============================================================================
# 3. 有多少種比賽層級？
# ==============================================================================
tourney_levels = matches["tourney_level"].dropna().unique().tolist()
print(f"\n比賽層級共有 {len(tourney_levels)} 種: {tourney_levels}\n")

# ==============================================================================
# 4. 有多少場的 winner 身高比 loser 高 / 相等 / 低？
# ==============================================================================
# 排除雙方任一身高欄位為 NaN 的場次
valid_ht = matches.dropna(subset=["winner_ht", "loser_ht"]).copy()

taller_count = (valid_ht["winner_ht"] > valid_ht["loser_ht"]).sum()
equal_count = (valid_ht["winner_ht"] == valid_ht["loser_ht"]).sum()
shorter_count = (valid_ht["winner_ht"] < valid_ht["loser_ht"]).sum()
total_valid = len(valid_ht)

print("身高與勝負關係（排除缺漏值）:")
print(f"- Winner 比 Loser 高: {taller_count:,} 場 ({taller_count / total_valid:.2%})")
print(f"- Winner 與 Loser 等高: {equal_count:,} 場 ({equal_count / total_valid:.2%})")
print(f"- Winner 比 Loser 矮: {shorter_count:,} 場 ({shorter_count / total_valid:.2%})\n")

# ==============================================================================
# 5. 依照不同的 tourney_level 來統計身高和勝負間的關係
# ==============================================================================
level_ht_summary = []
for level, group in valid_ht.groupby("tourney_level"):
    g = (group["winner_ht"] > group["loser_ht"]).sum()
    e = (group["winner_ht"] == group["loser_ht"]).sum()
    l = (group["winner_ht"] < group["loser_ht"]).sum()
    tot = len(group)
    diff_tot = g + l  # 雙方有身高差的場次總數
    
    level_ht_summary.append({
        "tourney_level": level,
        "總場數": tot,
        "Winner較高": g,
        "雙方等高": e,
        "Winner較矮": l,
        "佔該層級比例": f"{g / tot:.2%}",
        "有身高差時較高者勝率": f"{g / diff_tot:.2%}" if diff_tot > 0 else "N/A"
    })

level_ht_df = pd.DataFrame(level_ht_summary)
print("各賽事層級 (tourney_level) 身高與勝負關係:")
print(level_ht_df.to_string(index=False))

讀取的比賽檔案: ['atp_matches_2021.csv', 'atp_matches_2022.csv', 'atp_matches_2023.csv', 'atp_matches_2024.csv']
總比賽場次: 11,712 場

TPE 球員勝負統計:
- 勝場: 17 場
- 敗場: 44 場

比賽層級共有 6 種: ['A', 'M', 'F', 'G', 'D', 'O']

身高與勝負關係（排除缺漏值）:
- Winner 比 Loser 高: 5,643 場 (48.80%)
- Winner 與 Loser 等高: 1,325 場 (11.46%)
- Winner 比 Loser 矮: 4,596 場 (39.74%)

各賽事層級 (tourney_level) 身高與勝負關係:
tourney_level  總場數  Winner較高  雙方等高  Winner較矮 佔該層級比例 有身高差時較高者勝率
            A 6295      3023   727      2545 48.02%     54.29%
            D  677       303    90       284 44.76%     51.62%
            F   75        33    10        32 44.00%     50.77%
            G 2031      1039   225       767 51.16%     57.53%
            M 2422      1216   269       937 50.21%     56.48%
            O   64        29     4        31 45.31%     48.33%


### 先將年齡 .round(),
- value_counts winner 的年紀。

In [3]:
# 1. 勝者年齡四捨五入並統計分佈（獨立過濾 winner_age 缺漏值）
valid_winner_age = matches.dropna(subset=["winner_age"]).copy()
valid_winner_age["winner_age_rnd"] = valid_winner_age["winner_age"].round().astype(int)

print(f"Winner 年齡完整分佈 (共 {len(valid_winner_age):,} 場有效勝者紀錄)：")
print(valid_winner_age["winner_age_rnd"].value_counts().sort_index())

Winner 年齡完整分佈 (共 11,710 場有效勝者紀錄)：
winner_age_rnd
17       6
18      82
19     240
20     425
21     525
22     812
23     792
24    1180
25    1125
26    1271
27    1071
28     846
29     541
30     545
31     383
32     364
33     321
34     367
35     270
36     260
37     136
38      98
39      14
40      24
41       1
42       7
43       1
44       2
45       1
Name: count, dtype: int64


- 根據球員的年齡，統計勝率。

In [4]:
# 2. 敗者年齡四捨五入 (獨立過濾 loser_age 缺漏值)
valid_loser_age = matches.dropna(subset=["loser_age"]).copy()
valid_loser_age["loser_age_rnd"] = valid_loser_age["loser_age"].round().astype(int)

# 分別計算勝場與敗場
w_by_age = valid_winner_age["winner_age_rnd"].value_counts()
l_by_age = valid_loser_age["loser_age_rnd"].value_counts()

# 合併為完整統計表
all_age_stats = pd.DataFrame({"勝場": w_by_age, "敗場": l_by_age}).fillna(0).astype(int)
all_age_stats["總場數"] = all_age_stats["勝場"] + all_age_stats["敗場"]
all_age_stats["勝率數值"] = all_age_stats["勝場"] / all_age_stats["總場數"]
all_age_stats["勝率"] = all_age_stats["勝率數值"].map("{:.2%}".format)
all_age_stats = all_age_stats.sort_index()

print("所有年齡之勝率統計表 (修正單邊缺值後)：")
print(all_age_stats[["勝場", "敗場", "總場數", "勝率"]].to_string())

# 檢驗修正後的五個關鍵年齡勝敗場數
print("\n關鍵修正年齡確認 (18, 19, 21, 24, 27):")
print(all_age_stats.loc[[18, 19, 21, 24, 27], ["勝場", "敗場", "總場數", "勝率"]])

所有年齡之勝率統計表 (修正單邊缺值後)：
      勝場    敗場   總場數       勝率
15     0     1     1    0.00%
16     0     6     6    0.00%
17     6    21    27   22.22%
18    82    97   179   45.81%
19   240   214   454   52.86%
20   425   378   803   52.93%
21   525   419   944   55.61%
22   812   722  1534   52.93%
23   792   707  1499   52.84%
24  1180  1076  2256   52.30%
25  1125   982  2107   53.39%
26  1271  1179  2450   51.88%
27  1071   995  2066   51.84%
28   846   965  1811   46.71%
29   541   695  1236   43.77%
30   545   680  1225   44.49%
31   383   521   904   42.37%
32   364   504   868   41.94%
33   321   371   692   46.39%
34   367   344   711   51.62%
35   270   238   508   53.15%
36   260   235   495   52.53%
37   136   152   288   47.22%
38    98   122   220   44.55%
39    14    31    45   31.11%
40    24    33    57   42.11%
41     1    10    11    9.09%
42     7     8    15   46.67%
43     1     1     2   50.00%
44     2     2     4   50.00%
45     1     0     1  100.00%

關鍵修正年齡確認 (18, 19,

- 列出勝場數最多的前十位球員。

In [5]:
# 4. 列出勝場數最多的前 10 位球員
print("【勝場數最多的前 10 位球員】")
print(matches["winner_name"].value_counts().head(10))
print("-" * 50)

【勝場數最多的前 10 位球員】
winner_name
Jannik Sinner         237
Daniil Medvedev       221
Alexander Zverev      213
Stefanos Tsitsipas    212
Carlos Alcaraz        208
Andrey Rublev         205
Casper Ruud           197
Novak Djokovic        193
Taylor Fritz          187
Alex De Minaur        168
Name: count, dtype: int64
--------------------------------------------------


- 列出勝率最高的前十位球員。

In [6]:
import numpy as np
import pandas as pd

# 建立 ID 與姓名對照字典 (以最新或出現過的姓名為主)
id_to_name = pd.concat([
    matches[["winner_id", "winner_name"]].rename(columns={"winner_id": "id", "winner_name": "name"}),
    matches[["loser_id", "loser_name"]].rename(columns={"loser_id": "id", "loser_name": "name"})
]).drop_duplicates(subset=["id"]).set_index("id")["name"].to_dict()

# 依 player_id 統計勝負場次
p_wins = matches["winner_id"].value_counts()
p_losses = matches["loser_id"].value_counts()

player_stats = pd.DataFrame({"勝場": p_wins, "敗場": p_losses}).fillna(0).astype(int)
player_stats["總場次"] = player_stats["勝場"] + player_stats["敗場"]
player_stats["勝率數值"] = player_stats["勝場"] / player_stats["總場次"]
player_stats["球員姓名"] = player_stats.index.map(id_to_name)

print("【勝率最高的前 10 位球員（版本 A：完全不設門檻，純字面題意）】")
top10_no_thresh = player_stats.sort_values(by=["勝率數值", "勝場"], ascending=False).head(10).copy()
top10_no_thresh["勝率"] = top10_no_thresh["勝率數值"].map("{:.2%}".format)
print(top10_no_thresh[["球員姓名", "勝場", "敗場", "總場次", "勝率"]])
print("註：未設門檻時，前數名皆為僅出賽 1~2 場且未嘗敗績之球員（樣本偏差）。\n")

print("【勝率最高的前 10 位球員（版本 B：設定總出賽 >= 30 場以排除極端值）】")
top10_with_thresh = player_stats[player_stats["總場次"] >= 30].sort_values(by="勝率數值", ascending=False).head(10).copy()
top10_with_thresh["勝率"] = top10_with_thresh["勝率數值"].map("{:.2%}".format)
print(top10_with_thresh[["球員姓名", "勝場", "敗場", "總場次", "勝率"]])
print("註：設定出賽場次 >= 30 場為自訂補充條件，旨在反映具巡迴賽常規代表性之頂尖選手表現。\n")

【勝率最高的前 10 位球員（版本 A：完全不設門檻，純字面題意）】
                       球員姓名  勝場  敗場  總場次       勝率
122533         Nam Hoang Ly   4   0    4  100.00%
206470          Cezar Cretu   2   0    2  100.00%
210142     Olaf Pieczkowski   2   0    2  100.00%
210312       Oliver Ojakaar   2   0    2  100.00%
103917        Nicolas Mahut   1   0    1  100.00%
104235     Matwe Middelkoop   1   0    1  100.00%
105271      Gonzalo Escobar   1   0    1  100.00%
105434        Ruan Roelofse   1   0    1  100.00%
105464  Christopher Rungkat   1   0    1  100.00%
105587   Tom Kocevar Desman   1   0    1  100.00%
註：未設門檻時，前數名皆為僅出賽 1~2 場且未嘗敗績之球員（樣本偏差）。

【勝率最高的前 10 位球員（版本 B：設定總出賽 >= 30 場以排除極端值）】
                      球員姓名   勝場  敗場  總場次      勝率
104925      Novak Djokovic  193  31  224  86.16%
207989      Carlos Alcaraz  208  55  263  79.09%
206173       Jannik Sinner  237  64  301  78.74%
106421     Daniil Medvedev  221  71  292  75.68%
104745        Rafael Nadal   77  25  102  75.49%
100644    Alexander Zverev  213  74  287

- 先將年齡 .round(),
    - 製作一個表格來統計不同年紀球員之間的對戰數量， index 是 winner 年齡, column 是 loser 年齡，表格中的數值是對戰場數。只統計年齡 19-28。

In [7]:
# 重新定義雙邊年齡皆有效的資料集 (對戰矩陣專用)
valid_age = matches.dropna(subset=["winner_age", "loser_age"]).copy()
valid_age["winner_age_rnd"] = valid_age["winner_age"].round().astype(int)
valid_age["loser_age_rnd"] = valid_age["loser_age"].round().astype(int)

# 篩選雙方皆為 19 至 28 歲的賽事
age_19_28 = valid_age[
    valid_age["winner_age_rnd"].between(19, 28) & 
    valid_age["loser_age_rnd"].between(19, 28)
]

# 製作對戰場數矩陣 (Index: 勝者年齡, Column: 敗者年齡)
match_counts = pd.crosstab(
    age_19_28["winner_age_rnd"], 
    age_19_28["loser_age_rnd"],
    rownames=["勝者年齡 (Winner)"],
    colnames=["敗者年齡 (Loser)"]
).reindex(index=range(19, 29), columns=range(19, 29), fill_value=0)

print("【19–28 歲球員對戰場數表格】")
match_counts

【19–28 歲球員對戰場數表格】


敗者年齡 (Loser),19,20,21,22,23,24,25,26,27,28
勝者年齡 (Winner),,,,,,,,,,
19,3,11,12,15,16,22,24,23,16,19
20,11,19,16,33,30,29,33,47,54,25
21,12,14,22,32,30,50,40,71,35,49
22,12,22,29,44,46,65,62,81,78,77
23,6,28,32,34,55,64,66,71,85,69
24,18,37,43,82,58,117,104,115,95,84
25,17,29,46,73,65,115,91,107,94,100
26,31,41,47,73,79,135,102,115,107,115
27,27,30,38,70,78,91,95,117,92,96


- 先將年齡 .round(),
    - 製作一個表格來統計不同年紀球員之間的對戰數量， index 是球員年齡, column 是對手年齡，表格中的數值是勝率。
    - 有哪些年齡層，對於其他年齡層的勝率都是 49% 以上的？ (只考慮年齡 19-28)

In [8]:
age_range = range(19, 29)
age_19_28 = valid_age[
    valid_age["winner_age_rnd"].between(19, 28) & 
    valid_age["loser_age_rnd"].between(19, 28)
]

# 對戰數矩陣
match_counts = pd.crosstab(
    age_19_28["winner_age_rnd"], 
    age_19_28["loser_age_rnd"],
    rownames=["勝者年齡 (Winner)"],
    colnames=["敗者年齡 (Loser)"]
).reindex(index=age_range, columns=age_range, fill_value=0)

# 勝率矩陣
total_encounters = match_counts + match_counts.T
win_rate_matrix = pd.DataFrame(index=age_range, columns=age_range, dtype=float)

for a1 in age_range:
    for a2 in age_range:
        if a1 == a2:
            win_rate_matrix.loc[a1, a2] = 0.500  # 同年齡相殘固定為 50%
        else:
            tot = total_encounters.loc[a1, a2]
            win_rate_matrix.loc[a1, a2] = match_counts.loc[a1, a2] / tot if tot > 0 else np.nan

print("【19–28 歲球員勝率矩陣（小數點後三位）】")
display(win_rate_matrix.round(3))

# 判定是否有年齡層對其他年齡層勝率皆高於 49% (> 0.49)
dominant_ages = []
for age in age_range:
    other_opponents = [x for x in age_range if x != age]
    rates = win_rate_matrix.loc[age, other_opponents].dropna()
    # 嚴格依題意使用 > 0.49
    if (rates > 0.49).all():
        dominant_ages.append(age)

print(f"\n對其他所有年齡層勝率皆「高於 49% (> 0.49)」的年齡層: {dominant_ages}")
print("結論分析：輸出為空集合 []。在 19–28 歲之各年齡層中，皆至少存在一個對手年齡層使其勝率未達 49%。")

【19–28 歲球員勝率矩陣（小數點後三位）】


,19,20,21,22,23,24,25,26,27,28
19,0.500,0.500,0.500,0.556,0.727,0.550,0.585,0.426,0.372,0.487
20,0.500,0.500,0.533,0.600,0.517,0.439,0.532,0.534,0.643,0.439
21,0.500,0.467,0.500,0.525,0.484,0.538,0.465,0.602,0.479,0.653
22,0.444,0.400,0.475,0.500,0.575,0.442,0.459,0.526,0.527,0.575
23,0.273,0.483,0.516,0.425,0.500,0.525,0.504,0.473,0.521,0.585
24,0.450,0.561,0.462,0.558,0.475,0.500,0.475,0.460,0.511,0.575
25,0.415,0.468,0.535,0.541,0.496,0.525,0.500,0.512,0.497,0.575
26,0.574,0.466,0.398,0.474,0.527,0.540,0.488,0.500,0.478,0.550
27,0.628,0.357,0.521,0.473,0.479,0.489,0.503,0.522,0.500,0.561
28,0.513,0.561,0.347,0.425,0.415,0.425,0.425,0.450,0.439,0.500



對其他所有年齡層勝率皆「高於 49% (> 0.49)」的年齡層: []
結論分析：輸出為空集合 []。在 19–28 歲之各年齡層中，皆至少存在一個對手年齡層使其勝率未達 49%。


## Task 4: 自訂五個問題與 Pandas 分析
請給出五個你自己提出的問題，並且利用 pandas 回答這五個問題：
1. **哪一個球員的出賽場次最多**
2. **被零封最多的是哪個國家**
3. **一年之中哪個月分比賽場次最多**
4. **左手持拍的勝率有高於右手持拍嗎**
5. **比賽最長的前五場比賽分別是多久**

In [9]:
# ------------------------------------------------------------------------------
# Q1. 哪一個球員的出賽場次最多？
# ------------------------------------------------------------------------------
print("\n--- [問題 1] 2021–2024 年間，哪一位球員出賽總場次最多？ ---")
# 以 player_id 為主鍵統計勝場與敗場
w_counts = matches["winner_id"].value_counts()
l_counts = matches["loser_id"].value_counts()

total_matches_player = pd.DataFrame({"勝場": w_counts, "敗場": l_counts}).fillna(0).astype(int)
total_matches_player["總出賽場次"] = total_matches_player["勝場"] + total_matches_player["敗場"]

# 建立 ID 與姓名對照表
id_name_map = pd.concat([
    matches[["winner_id", "winner_name"]].rename(columns={"winner_id": "id", "winner_name": "name"}),
    matches[["loser_id", "loser_name"]].rename(columns={"loser_id": "id", "loser_name": "name"})
]).drop_duplicates(subset=["id"]).set_index("id")["name"].to_dict()

total_matches_player["球員姓名"] = total_matches_player.index.map(id_name_map)

# 取得出賽場次前 5 名
top_played = total_matches_player.sort_values(by="總出賽場次", ascending=False).head(5)
print(top_played[["球員姓名", "勝場", "敗場", "總出賽場次"]].to_string(index=False))

# 動態判斷並列第一的情況
max_matches = top_played["總出賽場次"].max()
top_players_list = top_played[top_played["總出賽場次"] == max_matches]["球員姓名"].tolist()

if len(top_players_list) > 1:
    names_str = " 與 ".join(top_players_list)
    print(f"【分析結論】出賽場次最多的是 {names_str} 並列第一，在 2021–2024 年間各自出賽達 {max_matches} 場。")
else:
    print(f"【分析結論】出賽場次最多的球員是 {top_players_list[0]}，累計出賽達 {max_matches} 場。")

# ------------------------------------------------------------------------------
# Q2. 被零封最多的是哪個國家？
# ------------------------------------------------------------------------------
print("\n--- [問題 2] 被「零封（比分中至少一盤為 6-0）」最多的是哪一個國家的選手？ ---")
score_valid = matches.dropna(subset=["score", "loser_ioc"]).copy()
# 比分欄的每一盤以 winner-loser 順序記錄；只有 6-0 代表 loser 該盤 0 分。
bagel_matches = score_valid[score_valid["score"].str.contains(r"(?:^|\s)6-0(?:\(|\s|$)", regex=True)]

bagel_by_country = bagel_matches["loser_ioc"].value_counts().head(5).reset_index()
bagel_by_country.columns = ["國家代碼 (IOC)", "被零封場次 (次數)"]

print(bagel_by_country.to_string(index=False))
top_bagel_country = bagel_by_country.iloc[0]["國家代碼 (IOC)"]
top_bagel_count = bagel_by_country.iloc[0]["被零封場次 (次數)"]
print(f"【分析結論】被對手送出單盤 6-0 零封次數最多的國家是 {top_bagel_country}（共 {top_bagel_count} 次）。"
      "此結果主要反映該國於各層級賽事中具備極龐大的參賽球員基底與總出場場次。")

# ------------------------------------------------------------------------------
# Q3. 一年之中哪一個月份比賽場次最多？
# ------------------------------------------------------------------------------
print("\n--- [問題 3] 一年 12 個月份之中，哪一個月份舉行的比賽場次最多？ ---")
matches_date = matches.dropna(subset=["tourney_date"]).copy()
matches_date["month"] = matches_date["tourney_date"].astype(str).str[4:6].astype(int)

monthly_counts = matches_date["month"].value_counts().sort_index().reset_index()
monthly_counts.columns = ["月份", "總比賽場次"]
monthly_counts["月份名稱"] = monthly_counts["月份"].map(lambda m: f"{m:02d} 月")

print(monthly_counts[["月份名稱", "總比賽場次"]].sort_values(by="總比賽場次", ascending=False).to_string(index=False))

busiest_month = monthly_counts.sort_values(by="總比賽場次", ascending=False).iloc[0]
print(f"【分析結論】整年之中比賽場次最多的是 {busiest_month['月份名稱']}，共計 {busiest_month['總比賽場次']:,} 場比賽。"
      "（註：本統計以賽事週之 tourney_date 為基準，代表該賽事開賽所屬月份之總場次，適逢澳網後常規巡迴賽密集舉辦期。）")


--- [問題 1] 2021–2024 年間，哪一位球員出賽總場次最多？ ---
              球員姓名  勝場  敗場  總出賽場次
Stefanos Tsitsipas 212  89    301
     Jannik Sinner 237  64    301
     Andrey Rublev 205  95    300
   Daniil Medvedev 221  71    292
  Alexander Zverev 213  74    287
【分析結論】出賽場次最多的是 Stefanos Tsitsipas 與 Jannik Sinner 並列第一，在 2021–2024 年間各自出賽達 301 場。

--- [問題 2] 被「零封（比分中至少一盤為 6-0）」最多的是哪一個國家的選手？ ---
國家代碼 (IOC)  被零封場次 (次數)
       ESP          61
       FRA          53
       USA          53
       ARG          43
       ITA          42
【分析結論】被對手送出單盤 6-0 零封次數最多的國家是 ESP（共 61 次）。此結果主要反映該國於各層級賽事中具備極龐大的參賽球員基底與總出場場次。

--- [問題 3] 一年 12 個月份之中，哪一個月份舉行的比賽場次最多？ ---
月份名稱  總比賽場次
02 月   1521
07 月   1298
08 月   1249
05 月   1188
04 月   1142
10 月   1111
01 月   1006
03 月    961
09 月    921
06 月    907
11 月    383
12 月     25
【分析結論】整年之中比賽場次最多的是 02 月，共計 1,521 場比賽。（註：本統計以賽事週之 tourney_date 為基準，代表該賽事開賽所屬月份之總場次，適逢澳網後常規巡迴賽密集舉辦期。）


In [10]:
# ------------------------------------------------------------------------------
# Q4. 左手持拍的勝率有高於右手持拍嗎？
# ------------------------------------------------------------------------------
print("\n--- [問題 4] 左手持拍 (L) 與右手持拍 (R) 球員正面對抗時，各手性的獲勝場次比例為何？ ---")
# 篩選「左手 vs 右手」正面交手的賽事
lr_duel = matches[
    ((matches["winner_hand"] == "L") & (matches["loser_hand"] == "R")) |
    ((matches["winner_hand"] == "R") & (matches["loser_hand"] == "L"))
].copy()

total_lr_duels = len(lr_duel)
left_wins = (lr_duel["winner_hand"] == "L").sum()
right_wins = (lr_duel["winner_hand"] == "R").sum()
left_win_rate = left_wins / total_lr_duels
right_win_rate = right_wins / total_lr_duels

q4_df = pd.DataFrame({
    "持拍類型": ["左手持拍 (L)", "右手持拍 (R)"],
    "正面對決勝場": [left_wins, right_wins],
    "總對抗場次": [total_lr_duels, total_lr_duels],
    "獲勝場次比例": [f"{left_win_rate:.2%}", f"{right_win_rate:.2%}"]
})
print(q4_df.to_string(index=False))

print(f"【分析結論】否。在共 {total_lr_duels:,} 場「左手 vs 右手」正面交手的比賽中，"
      f"左手球員勝出 {left_wins:,} 場（佔 {left_win_rate:.2%}），右手球員勝出 {right_wins:,} 場（佔 {right_win_rate:.2%}）。"
      "此指標代表此類對戰組合中兩側選手獲勝的比賽場次比例，而非選手個人的總體勝率，亦無法直接作為持拍手性決定比賽勝負的因果依據。")

# ------------------------------------------------------------------------------
# Q5. 比賽最長的前五場比賽分別是多久？
# ------------------------------------------------------------------------------
print("\n--- [問題 5] 2021–2024 年間，比賽耗時最長（minutes）的前五場比賽分別是多久？ ---")
# 排除時間欄位缺失的場次，並取得有效樣本數與缺漏數
total_matches_all = len(matches)
valid_time = matches.dropna(subset=["minutes"]).copy()
missing_time_count = total_matches_all - len(valid_time)

# 提取年份與開賽日期便於辨識不同年度賽事
valid_time["year"] = valid_time["tourney_date"].astype(str).str[:4]
top5_longest = valid_time.sort_values(by="minutes", ascending=False).head(5)

q5_display = []
for _, row in top5_longest.iterrows():
    mins = int(row["minutes"])
    hrs = mins // 60
    rem_mins = mins % 60
    q5_display.append({
        "年份": row["year"],
        "賽事日期": row["tourney_date"],
        "賽事名稱": row["tourney_name"],
        "獲勝者": row["winner_name"],
        "落敗者": row["loser_name"],
        "比分": row["score"],
        "比賽耗時": f"{mins} 分鐘 ({hrs} 小時 {rem_mins} 分)"
    })

print(pd.DataFrame(q5_display).to_string(index=False))
print(f"（備註：全體 {total_matches_all:,} 場比賽中，共有 {missing_time_count:,} 場因無記錄比賽時間未納入排序）")

longest_match = q5_display[0]
print(f"【分析結論】在有記錄比賽時間的場次中，耗時最長的一場為 {longest_match['年份']} 年 {longest_match['賽事名稱']} "
      f"由 {longest_match['獲勝者']} 對決 {longest_match['落敗者']} 的大戰，總共鏖戰了 {longest_match['比賽耗時']}。")


--- [問題 4] 左手持拍 (L) 與右手持拍 (R) 球員正面對抗時，各手性的獲勝場次比例為何？ ---
    持拍類型  正面對決勝場  總對抗場次 獲勝場次比例
左手持拍 (L)    1281   2705 47.36%
右手持拍 (R)    1424   2705 52.64%
【分析結論】否。在共 2,705 場「左手 vs 右手」正面交手的比賽中，左手球員勝出 1,281 場（佔 47.36%），右手球員勝出 1,424 場（佔 52.64%）。此指標代表此類對戰組合中兩側選手獲勝的比賽場次比例，而非選手個人的總體勝率，亦無法直接作為持拍手性決定比賽勝負的因果依據。

--- [問題 5] 2021–2024 年間，比賽耗時最長（minutes）的前五場比賽分別是多久？ ---
  年份     賽事日期            賽事名稱             獲勝者                落敗者                           比分               比賽耗時
2023 20230116 Australian Open     Andy Murray Thanasi Kokkinakis    4-6 6-7(4) 7-6(5) 6-3 7-5 345 分鐘 (5 小時 45 分)
2024 20240826         Us Open    Daniel Evans    Karen Khachanov 6-7(6) 7-6(2) 7-6(4) 4-6 6-4 335 分鐘 (5 小時 35 分)
2023 20230529   Roland Garros Daniel Altmaier      Jannik Sinner 6-7(0) 7-6(7) 1-6 7-6(4) 7-5 326 分鐘 (5 小時 26 分)
2022 20220117 Australian Open    Rafael Nadal    Daniil Medvedev       2-6 6-7(5) 6-4 6-4 7-5 324 分鐘 (5 小時 24 分)
2022 20220829         Us Open  Carlos Alcaraz      Jannik Sinner    6-3 6-7(7) 